# MDP and Q-learning/SARSA

In this notebook, you will build on your previous understanding of the k-armed bandit problem, which involves choosing the best action without considering the history or future states. Unlike bandit algorithms, reinforcement learning methods like Q-learning and SARSA take into account the sequence of states and actions over time, capturing the environment’s dynamics through the concept of a Markov Decision Process (MDP).

**Key points of this notebook include:**  
- Introduction to the Markov Decision Process framework  
- Exploration of Q-learning and SARSA algorithms  
- Creation of an environment and an agent with an epsilon-greedy policy (with each class having a specific objective)  
- Implementation of epsilon decay to enable high exploration initially and minimal exploration later  
- Manual execution of a grid search to tune hyperparameters for better performance  

The objective of this notebook is to introduce the Markov decision process, the q-learning algorithm, the sarsa algorithm and compare the 2 algorithms.  

To do this, we'll use the game of dice-jack, a simplified version of blackjack with a die. The goal is to roll the die as many times as you want, adding up the score each time to get the highest score without going over 21.  


## 1. Markov decision process

A **Markov Decision Process (MDP)** is a mathematical framework for modeling decision-making in situations where outcomes are partly random and partly under the control of a decision maker.

### Key Components of an MDP:

1. **States (S)**: All possible situations the agent can be in
2. **Actions (A)**: All possible actions the agent can take
3. **Transition Probabilities (P)**: Probability of moving from one state to another given an action
4. **Rewards (R)**: Immediate reward received after taking an action in a state
5. **Policy (π)**: Strategy that defines what action to take in each state

### The Markov Property

The **Markov Property** states that the future depends only on the current state, not on the history of how we got there:

```
P(S_{t+1} | S_t, A_t, S_{t-1}, A_{t-1}, ..., S_0, A_0) = P(S_{t+1} | S_t, A_t)
```

This means: *"The future is independent of the past given the present."*

For more explanation : https://www.youtube.com/watch?v=Rgxs8lfoG4I&t=37s

**Exercise 1:**   
Now, draw a Markov process diagram for the dice-jack. Then create an environment for the dice-jack by following your Markov process diagram.


### CORRECTION

Présentation du jeu du dice-jack : ne pas dépasser 21 avec un dé à 6 faces tout en faisant le meilleur score.

Actions : lancer le dé ou s'arrêter

États : le score de la partie. état initial = 0, état terminal quand on choisit de s'arrêter ou qu'on dépasse 21

Rewards : le score de la partie ou -50 si on dépasse 21

Politique : epsilon greedy

In [1]:
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from itertools import product
import pickle
from datetime import datetime

# Argmax function for later in the notebook
def argmax(q_values: list) -> int:
    """
    Takes in a list of q_values and returns the index of the item
    with the highest value. Breaks ties randomly.
    returns: int - the index of the highest value in q_values
    """
    arr_q_values = np.array(q_values)
    return np.random.choice(np.where(arr_q_values == arr_q_values.max())[0])

In [8]:
NEGATIVE_REWARD = -50
LIMIT = 21
BUST_STATE = 22  # État spécial pour "dépassé 21"

class Black_dice_env:
    """
    A Blackjack-like dice game environment for reinforcement learning.
    
    The agent can either draw a dice (action "tirer") or stop (action "stop").
    The goal is to get as close as possible to the limit without exceeding it.
    If the score exceeds the limit, the agent receives a negative reward.
    """
    
    def __init__(self, limit=LIMIT):
        """
        Initialize the Black Dice environment.
        
        Args:
            limit (int): The maximum score allowed before busting. Default is LIMIT (21).
                        If the score exceeds this limit, the game ends with a negative reward.
        
        Attributes:
            limit (int): The maximum allowed score
            score (int): Current cumulative score, initialized to 0
        """
        self.limit = limit
        self.score = 0
        
    def step(self, action):
        """
        Execute one step in the environment based on the given action.
        
        Args:
            action (str): The action to take. Can be:
                         - "tirer": Draw a dice (roll a 6-sided die and add to score)
                         - "stop": Stop the game and receive reward equal to current score
        
        Returns:
            tuple: A tuple containing (next_state, reward, done) where:
                - next_state (int): The next state after the action
                    * Current score if action is valid and not busted
                    * BUST_STATE (22) if score exceeds the limit
                - reward (int): The reward received
                    * NEGATIVE_REWARD (-50) if busted (score > limit)
                    * Current score if action is "stop"
                    * 0 for intermediate "tirer" actions that don't bust
                - done (bool): Whether the episode has ended
                    * True if busted or if action is "stop"
                    * False for valid "tirer" actions that don't bust
        
        Behavior:
            - "tirer": Rolls a dice (1-6), adds to score. If score exceeds limit,
                     returns negative reward and ends episode. Otherwise continues.
            - "stop": Ends episode immediately with reward equal to current score.
        """
        if action == 'tirer':
            next_state = np.min(next_state +np.random.randint(1,7),BUST_STATE)
            reward = 0 if next_state<=self.limit else NEGATIVE_REWARD
            done = next_state==BUST_STATE

        elif action=='stop':
            next_state = self.score 
            reward = self.score 
            done = True

        return (next_state,reward,done)

    def reset(self):
        """
        Reset the environment to its initial state.
        
        This method resets the cumulative score to 0, preparing the environment
        for a new episode.
        
        Returns:
            int: The initial state (score = 0)
        
        Note:
            This method should be called at the beginning of each new episode
            to ensure the environment starts from a clean state.
        """
        self.score = 0

## 2. Q-learning

Q-learning and **k-armed bandit algorithms** both address the challenge of learning optimal actions based on rewards, but their scope and methods differ significantly. In the k-armed bandit problem, the agent chooses between multiple options (arms) and learns which arm yields the highest reward—the environment does not change state, so the task is to balance exploration (trying different arms) and exploitation (sticking with the best known arm). Typical algorithms for this setting include the greedy and ε-greedy approaches, which update the estimated value of each arm directly based on observed rewards, without considering transitions between different states.

Q-learning, on the other hand, is suited for more complex environments where actions lead to subsequent states and rewards may depend on the history of states and actions. It maintains a Q-table, where each entry $ Q(s,a) $ estimates the expected reward of taking action $ a $ in state $ s $ and following the optimal strategy thereafter. The update rule for the Q-table, called the Bellman equation, is:

$$
Q(s,a) \leftarrow Q(s,a) + \alpha \Big( r + \gamma \max_{a'} Q(s',a') - Q(s,a) \Big)
$$

where:
- $ \alpha $ is the learning rate,
- $ r $ is the reward received,
- $ \gamma $ is the discount factor for future rewards,
- $ s' $ is the new state after taking action $ a $,
- $ \max_{a'} Q(s',a') $ is the best Q-value obtainable from $ s' $.

This equation lets Q-learning iteratively refine its estimate for each state-action pair, considering both immediate and future potential rewards. In contrast, k-armed bandit algorithms only update direct action-value estimates since there are no states to transition between—making them simpler but less flexible for environments with dynamic state transitions.



**Exercise 2.1:**  
Now, using the experience gained from the k-armed bandit, build an agent with an epsilon-greedy policy and update the Q-table using the Bellman equation (with $ \alpha $ = 0.1 and $ \gamma $ = 0.9). Test your agent on the dice-jack environment to explore reinforcement learning in action.  
  
**Exercide 2.2:**  
Previously, we observed that in a stable environment, exploration is crucial at the start of training but becomes unnecessary later on. To address this, you can gradually decrease the epsilon value during training, allowing for high exploration at first and less as learning progresses. Once your agent is trained and tested, implement epsilon decay to refine your agent’s policy.  
  
**Exercice 2.3:**  
Now that you have a working agent, test the impact of the Bellman equation parameters: $ \alpha $ and $ \gamma $. If you haven't already, research the meaning of these two parameters. Then, implement a grid search to systematically test different combinations of $ \alpha $ and $ \gamma $ ($ \gamma $ = [0.8, 0.9, 0.95, 0.99], $ \alpha $ = [0.05, 0.1, 0.2, 0.3]) to find the optimal parameter pair for your agent.

In [ ]:
class EpsilonGreedy:
    def __init__(self, num_bandits, epsilon=0.1):
        """
        Initializes the Epsilon-Greedy Agent.

        Args:
            num_bandits (int): The number of bandits (actions) available in the environment.
            epsilon (float): The probability of selecting a random action (exploration rate). Defaults to 0.1.

        Initializes:
            - q_values (np.ndarray): An array to store the estimated value of each bandit, initialized to zeros.
            - arm_counts (np.ndarray): An array to track how many times each bandit has been selected, initialized to zeros.
        """
        self.q_table = np.zeros([23,2])
        self.epsilon = epsilon

    def act(self):

        if np.random.rand()>self.epsilon:
            return argmax(self.q_values)
        else:
            return np.random.randint(len(self.q_values))
        

    def learn(self, action, previous_state,reward,alpha=0.1,gamma=0.9):
        """
        Updates the estimated value of a bandit based on the received reward.

        Args:
            action (int): The index of the bandit chosen by the agent.
            reward (float): The reward received after taking the action.

        Updates:
            - The count of times the bandit (action) has been selected.
            - The estimated value of the selected bandit using the formula:
              Q(a) <- Q(a) + (1/N(a))(R - Q(a))
              where:
                - Q(a)is the estimated value of the action,
                - N(a) is the number of times the action has been taken,
                - R is the reward received.
        """
        if action =='tirer':
            act = 0
        elif action == 'stop':
            act = 1
        try:
            q_values = self.q_table[reward:reward+7]
        except:
            q_values = self.q_table[reward:-1]
        best_possible_next_reward = argmax(q_values)
        self.q_table[previous_state,act] += alpha*(reward+gamma*best_possible_next_reward-self.self.q_table[previous_state,act])

In [ ]:
# A plenty of cells to try some code...

## 3. SARSA
**Difference Between Q-learning and SARSA**

**1. Nature of the Algorithm**  
- **Q-learning:** An *off-policy* algorithm. It learns the optimal policy independently of the agent’s actual behavior during exploration.  
- **SARSA:** An *on-policy* algorithm. It learns the value of the policy actually followed by the agent, including exploratory actions.

**2. Update Rule**  
- **Q-learning:**  
$$
Q(s, a) \leftarrow Q(s, a) + \alpha \big[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \big]
$$  
It uses the best possible action in the next state to update the Q-value, regardless of the action the agent actually takes.

- **SARSA:**  
$$
Q(s, a) \leftarrow Q(s, a) + \alpha \big[ r + \gamma Q(s', a') - Q(s, a) \big]
$$  
It updates the Q-value based on the action that the agent actually takes in the next state, so the update depends on the current policy (e.g., epsilon-greedy).

**3. Practical Implications**  
- **Q-learning:**  
  - Tends to learn the optimal policy faster.  
  - May be less stable if exploration leads the agent into risky situations (e.g., “cliff walking”).  
  - Less cautious, as it ignores the risks associated with exploratory actions.

- **SARSA:**  
  - More conservative because it accounts for the consequences of exploratory actions.  
  - More stable in environments where exploration can be dangerous.  
  - Converges to the optimal policy *for the policy followed* (e.g., epsilon-greedy), which can sometimes be slightly suboptimal if exploration remains high.

**Exercice 3.1 :**  
Reuse the dice_jack environment and your agent, this time implementing the SARSA algorithm to explore on-policy reinforcement learning.  

**Exercice 3.2 :**  
Plot the results obtained using both the Q-learning and SARSA algorithms on the same graph. Analyze and discuss the differences between them.

In [ ]:
# Your code here

In [ ]:
# And also here

In [ ]:
# Why not here ?

In [ ]:
# Is needed, this cell is also avalaible for your code

Conclusion: You may notice little or no difference between Q-learning and SARSA in your simplified blackjack environment with a six-sided die. 

This is because the rewards and transitions do not involve major risks—there is no strong penalty for exploratory actions. In such environments, both SARSA and Q-learning often converge to the same policy.

The distinction between these algorithms becomes more apparent in environments where exploration can lead to dangerous or costly outcomes.

To truly observe the difference, you need a setting where exploration carries risks. Fortunately, a maze environment has been prepared for you to explore this concept further.

## To go further: maze

I have created a maze environment so you can test the difference between SARSA and Q-learning. You can download the code here: (https://github.com/V1centFaure/maze/tree/main/src).

A quick note: I originally planned to make this available as a library for easy import, but ran into an issue. Please use only the files in the src folder to run the environment and the associated agent.

Enjoy !